<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# Depth Anything: Estimativa de Profundidade Monocular

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

Neste notebook, vamos usar o **Depth Anything V2** via HuggingFace Transformers para gerar mapas de profundidade a partir de imagens RGB.

Diferente da abordagem do artigo (que clona o repositorio original e usa Poetry/Streamlit), aqui usamos o pipeline do Transformers para inferencia direta.

In [ ]:
!pip install transformers torch torchvision matplotlib Pillow -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import urllib.request
import os

## Carregar o Modelo Depth Anything V2

Vamos usar o encoder **Small (vits)**, que e mais leve e roda bem em CPUs e GPUs do Colab.

In [ ]:
model_name = "depth-anything/Depth-Anything-V2-Small-hf"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForDepthEstimation.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Modelo carregado no dispositivo: {device}")

## Baixar Imagem de Exemplo

Vamos usar uma imagem de cena urbana para testar a estimativa de profundidade.

In [ ]:
# Baixar imagem de exemplo
url = "https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/Depth-Anything/sample_city.jpg"
filepath = "sample_city.jpg"

if not os.path.exists(filepath):
    urllib.request.urlretrieve(url, filepath)

image = Image.open(filepath)
plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title("Imagem de entrada")
plt.axis("off")
plt.show()

## Inferencia: Gerar Mapa de Profundidade

Passamos a imagem pelo processador e pelo modelo para obter o mapa de profundidade previsto.

In [ ]:
def predict_depth(image, processor, model, device):
    """Gera o mapa de profundidade para uma imagem PIL."""
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth

    # Interpolar para o tamanho original
    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=image.size[::-1],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

    return prediction.cpu().numpy()


depth_map = predict_depth(image, processor, model, device)

# Visualizar resultado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.imshow(image)
ax1.set_title("Imagem Original", fontsize=13)
ax1.axis("off")

ax2.imshow(depth_map, cmap="inferno")
ax2.set_title("Mapa de Profundidade", fontsize=13)
ax2.axis("off")

plt.tight_layout()
plt.show()

## Comparando Encoders: Small vs Large

O Depth Anything V2 oferece diferentes encoders. Vamos comparar o **Small (vits)** com o **Large (vitl)** para ver a diferenca de qualidade.

In [ ]:
# Carregar o modelo Large
model_large_name = "depth-anything/Depth-Anything-V2-Large-hf"

processor_large = AutoImageProcessor.from_pretrained(model_large_name)
model_large = AutoModelForDepthEstimation.from_pretrained(model_large_name)
model_large = model_large.to(device)

# Inferencia com o modelo Large
depth_map_large = predict_depth(image, processor_large, model_large, device)

# Comparacao lado a lado
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].imshow(image)
axes[0].set_title("Imagem Original", fontsize=13)
axes[0].axis("off")

axes[1].imshow(depth_map, cmap="inferno")
axes[1].set_title("Depth Anything V2 - Small (vits)", fontsize=13)
axes[1].axis("off")

axes[2].imshow(depth_map_large, cmap="inferno")
axes[2].set_title("Depth Anything V2 - Large (vitl)", fontsize=13)
axes[2].axis("off")

plt.suptitle("Comparacao entre encoders", fontsize=15)
plt.tight_layout()
plt.show()

## Processar Multiplas Imagens

Vamos baixar mais imagens e gerar mapas de profundidade para cada uma delas.

In [ ]:
# Baixar imagens adicionais
additional_urls = [
    ("https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/Depth-Anything/sample_indoor.jpg", "sample_indoor.jpg"),
    ("https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/Depth-Anything/sample_nature.jpg", "sample_nature.jpg"),
]

images = []
for url, name in additional_urls:
    if not os.path.exists(name):
        urllib.request.urlretrieve(url, name)
    images.append((name, Image.open(name)))

# Gerar mapas de profundidade
fig, axes = plt.subplots(len(images), 2, figsize=(16, 6 * len(images)))

for i, (name, img) in enumerate(images):
    depth = predict_depth(img, processor, model, device)

    axes[i][0].imshow(img)
    axes[i][0].set_title(f"Original: {name}", fontsize=12)
    axes[i][0].axis("off")

    axes[i][1].imshow(depth, cmap="inferno")
    axes[i][1].set_title("Mapa de Profundidade", fontsize=12)
    axes[i][1].axis("off")

plt.tight_layout()
plt.show()